# Encrypted BERT inference

Sentiment classification under CKKS. `perseus.nn.EncBert` runs the 12 encoder blocks on under a fixed bootstrap schedule.

*Client-side: the embeddings, and the head (pooler + `tanh` + classifier) as `tanh` has no
FHE approximation supported in our library yet.

<!-- prereq-cell -->
> **Prerequisites.** This notebook loads artifacts it does not create: the exported
> weights (`weights.bin.zip`, `client.npz`), the calibrated `configs.json`, and, for the
> planned path, a bootstrap plan. Run **`setup_artifacts.ipynb`** first if you do not
> have them.

In [ ]:
import os, time

import numpy as np
import torch                                   # torch before _core (NCCL load order)
from transformers import AutoTokenizer

from perseus import _core
from perseus.hub import load_model
from perseus.nn import EncBert

# ── write your sentence here ──────────────────────────────────────────────────
TEXT = "a masterpiece of modern cinema ."
# ──────────────────────────────────────────────────────────────────────────────
SENTIMENT = {0: "negative", 1: "positive"}

k = int(os.environ.get("GATE_BLOCKS", "12"))
model_name = os.environ.get("BERT_MODEL", "textattack/bert-base-uncased-SST-2") # specify the hf string directly.
model_dir = os.environ["BERT_MODEL_DIR"]

hf = load_model(model_name, device="cpu")
tok = AutoTokenizer.from_pretrained(model_name)
ids = tok(TEXT, return_tensors="pt")["input_ids"]

# plaintext reference logits (for comparison with encrypted inference)
with torch.no_grad():
    emb = hf.bert.embeddings(input_ids=ids, token_type_ids=torch.zeros_like(ids))
    h = emb
    for b in range(k):
        out = hf.bert.encoder.layer[b](h)
        h = out[0] if isinstance(out, tuple) else out   # BertLayer returns Tensor or tuple
    ref_logits = hf.classifier(hf.bert.pooler(h)).numpy()[0]

tokens = emb[0].numpy().tolist()
ref_id = int(np.argmax(ref_logits))
print(f"sentence : {TEXT!r}")
print(f"tokens   : {tok.convert_ids_to_tokens(ids[0])}")
print(f"plaintext: {SENTIMENT[ref_id]}")

## Build the encrypted model

In [ ]:
opts = _core.InferenceOptions()
opts.ckks = _core.CKKSOptions.from_env()
_mode = os.environ.get("BERT_MODE", "threaded") # "threaded" or "sync" for weights loading policy.
opts.mode = {"sync": _core.InferenceMode.Sync,
             "threaded": _core.InferenceMode.Threaded}[_mode]
inf = _core.make_bert_inference(opts) # collects rotation indexes and other params.

store = _core.WeightStore.from_zip(os.path.join(model_dir, "weights.bin.zip"))
configs = _core.load_configs(os.environ["CONFIGS_PATH"]) # must be generated with the setup_artifacts notebook.
model = EncBert(store, configs, n_layers=k).bind(inf) # inits a BERT model given a weight store and configs.

plan = os.environ.get("FHE_BOOTSTRAP_PLACEMENTS_DIR", "(eager)")
print(f"EncBert bound: {k} blocks   plan = {os.path.basename(plan)}   mode = {_mode}")

## Encrypted forward

The real arm we are proving can carry up to `slots / hidDim` = 32 tokens at the same time. Swithching to complex token pair raises the limit to 64 tokens.

In [ ]:
cap = inf.slots // inf.size.hidDim * (2 if inf.token_pair else 1)
# this is not an hard limit, but a practical for this notebook purpose.
if len(tokens) > cap:
    raise RuntimeError(f"T={len(tokens)} > {cap} = one packed chunk; shorten TEXT.")

xs, ns, ns_im = model.encode_tokens(tokens)
t0 = time.perf_counter()
# forward pass.
cls = np.array(model.decode_cls(model.forward(xs, ns, ns_im)))
print(f"forward: {time.perf_counter() - t0:.1f} s   ->  CLS vector of {cls.shape[0]} features")

## The prediction

Client head on the decrypted CLS: pooler dense, `tanh`, classifier.

In [ ]:
# plaintext reference logits
c = np.load(os.path.join(model_dir, "client.npz"))
# compute logits from the CLS ciphertext (tanh not supported yet)
logits = c["classifier_weight"] @ np.tanh(
    c["pooler_weight"] @ cls + c["pooler_bias"]) + c["classifier_bias"]
top1 = int(np.argmax(logits))
w_mape = np.abs(logits - ref_logits).sum() / np.abs(ref_logits).sum()

print(f"sentence  : {TEXT!r}")
print(f"encrypted : {SENTIMENT[top1]}   {np.round(logits, 3).tolist()}")
print(f"plaintext : {SENTIMENT[ref_id]}   {np.round(ref_logits, 3).tolist()}   "
      f"{'MATCH' if top1 == ref_id else 'DIFF'}")
print(f"w_mape    : {w_mape:.3f}")
assert np.isfinite(logits).all()
assert top1 == ref_id, f"encrypted {top1} != plaintext {ref_id}"

## Notes

- Change `TEXT` above; up to 32 tokens.